# Pipeline de Benchmark do TCC

Roda o sistema de benchmark (`benchmarks/`) que usa o **pipeline real**
(`TrainingService → ModelLoader → Predictor → MetricsCalculator`) e gera
tabelas LaTeX + figuras + JSON/CSV prontos para a monografia.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
while not (ROOT / "app").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Projeto:", ROOT)

## 1. Benchmark rápido (sintético) — valida o harness em segundos

In [ ]:
from benchmarks import BenchmarkConfig, run_benchmark

cfg = BenchmarkConfig.quick(
    architectures=["MultiscaleCNN", "SVM"],
    synthetic_n=160,
    snr_levels_db=[20],
    output_dir=str(ROOT / "results" / "notebook_benchmark"),
)
results = run_benchmark(cfg)

for name, r in results["architectures"].items():
    if r.get("status") == "ok":
        c = r["clean"]
        print(f"{name:<16} acc={c['accuracy']*100:5.1f}%  "
              f"AUC={c.get('auc_roc', float('nan')):.3f}  "
              f"conv={r.get('converged')}")
    else:
        print(f"{name:<16} ERRO: {r.get('error')}")

In [ ]:
# Artefatos gerados (tabelas .tex, figuras .png, CSV/JSON).
out = ROOT / "results" / "notebook_benchmark"
for f in sorted(out.rglob("*")):
    if f.is_file():
        print(f.relative_to(out))

## 2. Execução completa do TCC

Para o experimento oficial (download + processamento + treino + métricas),
use o automador de linha de comando (fora do notebook, pois é demorado):

```bash
python scripts/run_tcc_pipeline.py --tcc-full-dataset \
    --out results/tcc_full_20k \
    --npz app/datasets/benchmark_audio_raw_20k.npz
```

Ou o benchmark direto sobre um `.npz` já exportado:

```bash
python scripts/run_benchmark.py --full --dataset app/datasets/brspeech_df.npz
```

Veja `docs/15_BENCHMARK.md` para o mapeamento saída → tabela/figura do TCC.